In [0]:
dbutils.widgets.text("bronze_layer", "bronze")
bronze = dbutils.widgets.get("bronze_layer")

dbutils.widgets.text("silver_layer", "silver")
silver = dbutils.widgets.get("silver_layer")

dbutils.widgets.text("schema_name", "fhir")
schema = dbutils.widgets.get("schema_name")

dbutils.widgets.text("tables", "Patient,Encounter,Observation,Condition")
tables_str = dbutils.widgets.get("tables")

Tables = [t.strip() for t in tables_str.split(",")]

In [0]:
table = None
for i in Tables:
    if i.lower() == "observation":
        table = i
        break

if table is None:
    dbutils.notebook.exit("Exiting notebook: Table is not condition")

print(table + " table is present ")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Assigning the value to the variable and loading the data into df based on if the bronze table exist or not.

In [0]:
bronze_table = f"{bronze}.{schema}.{table}"
silver_table = f"{silver}.{schema}.{table}"

if spark.catalog.tableExists(silver_table):
    bronze_df = spark.table(bronze_table).filter(
        col("load_date") >= date_sub(current_date(), 4)
    )
else:
    bronze_df = spark.table(bronze_table)

bronze_df.select("load_date").show()

### Exploding and flattening the raw_JSON from df created from bronze table.


In [0]:

bronze_json_schema = """
array<
  struct<
    entry: array<
      struct<
        resource: struct<
          id: string,
          status: string,
          subject: struct<reference: string>,
          effectiveDateTime: string,
          valueString: string,
          valueQuantity: struct<value: double, unit: string>,
          code: struct<
            text: string,
            coding: array<struct<code: string, display: string>>
          >,
          category: array<
            struct<
              coding: array<struct<code: string, display: string>>
            >
          >,
          meta: struct<versionId: string, lastUpdated: string>
        >
      >
    >
  >
>
"""

parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_json"), bronze_json_schema))

exploded_df = parsed_df.select(
    explode(col("json_data")).alias("bundle"),
    "extraction_timestamp"
)

entry_df = exploded_df.select(
    explode(col("bundle.entry")).alias("entry"),
    "extraction_timestamp"
)

In [0]:
staging_df = entry_df.select(
    col("entry.resource.id").alias("observation_id"),
    col("entry.resource.status").alias("status"),

    # Patient
    col("entry.resource.subject.reference").alias("patient_ref"),

    # Code
    col("entry.resource.code.text").alias("observation_text"),
    col("entry.resource.code.coding")[0]["code"].alias("loinc_code"),
    col("entry.resource.code.coding")[0]["display"].alias("loinc_display"),

    # Category
    col("entry.resource.category")[0]["coding"][0]["code"].alias("category_code"),

    # Value fields
    col("entry.resource.valueString").alias("value_string"),
    col("entry.resource.valueQuantity.value").alias("value_quantity"),
    col("entry.resource.valueQuantity.unit").alias("value_unit"),

    # Time
    col("entry.resource.effectiveDateTime").alias("effective_datetime"),

    # Metadata
    col("entry.resource.meta.versionId").alias("version_id"),
    col("entry.resource.meta.lastUpdated").alias("last_updated"),

    col("extraction_timestamp")
)
staging_df.show(1)

In [0]:
staging_df = staging_df \
    .withColumn("patient_id", split(col("patient_ref"), "/")[1]) \
    .drop("patient_ref")

In [0]:
staging_df = staging_df \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

staging_df.orderBy("last_updated").show(1)

### Removing duplicate PK records based on last_updated date. So, only records having latest last_updated date will be considered.

In [0]:
window = Window.partitionBy("observation_id").orderBy(col("last_updated").desc())

staging_df = staging_df \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")

staging_df.show(1)

### Checking if silver table exist or not if it exist than the the code will go to merge in case if silver table now exist df will directly created the table and exit the notebook

In [0]:
silver_table = f"{silver}.{schema}.{table}"
print(silver_table)

if not spark.catalog.tableExists(silver_table):
    staging_df.write.format("delta").saveAsTable(silver_table)
    dbutils.notebook.exit("Exiting notebook: As this was the first time load")

In [0]:
staging_df.createOrReplaceTempView("staging_observation")

### SCD 2 Type loading data into silver table. Therefore, 3 things are handle - 
1. if an update is found for a record it will inactive the older record. 
2. if a new record is found it will insert it. 
3. and than it will insert the update record.

In [0]:
query = f"""
MERGE INTO {silver_table} AS tgt
USING staging_observation AS src
ON tgt.observation_id = src.observation_id
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.status <> src.status OR
    tgt.patient_id <> src.patient_id OR
    tgt.loinc_code <> src.loinc_code OR
    tgt.category_code <> src.category_code OR
    tgt.value_string <> src.value_string OR
    tgt.value_quantity <> src.value_quantity OR
    tgt.value_unit <> src.value_unit OR
    tgt.effective_datetime <> src.effective_datetime
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    observation_id,
    status,
    patient_id,
    loinc_code,
    category_code,
    value_string,
    value_quantity,
    value_unit,
    effective_datetime,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.observation_id,
    src.status,
    src.patient_id,
    src.loinc_code,
    src.category_code,
    src.value_string,
    src.value_quantity,
    src.value_unit,
    src.effective_datetime,
    src.last_updated,
    current_timestamp(),
    NULL,
    true
)
"""

query1 = f"""
INSERT INTO {silver_table} (
    observation_id,
    status,
    patient_id,
    loinc_code,
    category_code,
    value_string,
    value_quantity,
    value_unit,
    effective_datetime,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT 
    src.observation_id,
    src.status,
    src.patient_id,
    src.loinc_code,
    src.category_code,
    src.value_string,
    src.value_quantity,
    src.value_unit,
    src.effective_datetime,
    src.last_updated,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_observation src
JOIN {silver_table} tgt
    ON tgt.observation_id = src.observation_id
WHERE 
    tgt.is_current = false
AND (
    tgt.status <> src.status OR
    tgt.patient_id <> src.patient_id OR
    tgt.loinc_code <> src.loinc_code OR
    tgt.category_code <> src.category_code OR
    tgt.value_string <> src.value_string OR
    tgt.value_quantity <> src.value_quantity OR
    tgt.value_unit <> src.value_unit OR
    tgt.effective_datetime <> src.effective_datetime
)
AND NOT EXISTS (
    SELECT 1
    FROM {silver_table} t2
    WHERE t2.observation_id = src.observation_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)